In [167]:
import random
import copy

In [168]:
COLOURS = ['Red', 'Blue', 'Green', 'Yellow']
NUMBERS = list(range(10))         
PLAYERS = ['p1', 'p2', 'p3']  
PLAYER_NAMES = {'p1': 'P1 (Minimax Defensive)',
    'p2': 'P2 (Expectimax Offensive)' ,
    'p3': 'P3',
}

DEFAULT_SEED = 42

In [169]:
class Card:# single uno card

    def __init__(self, colour, value):
        self.colour = colour
        self.value = value           

        
    def __repr__(self):
        return f"{self.colour} {self.value}"

    def __eq__(self, other):
        return (isinstance(other, Card)and self.colour == other.colour and self.value == other.value)

    def __hash__(self):
        return hash((self.colour, self.value))

    def __deepcopy__(self, memo):
        #colour value dono fixed tu no need to recurse
        return Card(self.colour, self.value)



    def matches(self, top: 'Card'): #same colour or number
        return self.colour == top.colour or self.value == top.value
        
    def is_skip(self):
        return self.value == 'Skip'

#DEBUG
c1 = Card('Red' , 8)
print(c1.is_skip())

c2 = Card('Yellow ', 'Skip')
print(c2.is_skip())

c1.__repr__()

False
True


'Red 8'

In [170]:
def deck_generator(): #total 4×11 so 44cards 0 say 9 sab mai and 1 skip bhi so 11 

    deck = []
    for colour in COLOURS:
        for num in NUMBERS:                  
            deck.append(Card(colour, num))
        deck.append(Card(colour, 'Skip'))   

        
    random.shuffle(deck)
    return deck



#DEBUG
#deck= deck_generator()
#print(deck)
#len(deck)

In [171]:
def get_valid_moves(hand, top_card):
    return [card for card in hand if card.matches(top_card)]

#DEBUG
hand = [Card('Red', 6) , Card('Blue', 3), Card('Yellow', 4) , Card('Yellow', 'Skip') , Card('Green', 'Skip') ]
top = Card('Yellow', 6)
get_valid_moves(hand , top)

[Red 6, Yellow 4, Yellow Skip]

In [172]:
def apply_move(state, move):#player key is Player Names ki keys p1 ,p2 , p3

    player_id, card = move
    
    state_updated = copy.deepcopy(state)#deep copy takay original wala is not mutated
    skip_player= None

    if card is None:#card nai hai valid also reshuffle already played ya discards into deck agar deck hi khali hogaya
        if not state_updated['deck'] and state_updated.get('discards'):
            state_updated['deck'] = state_updated['discards']
            random.shuffle(state_updated['deck'])
            state_updated['discards'] = []

        
        if state_updated['deck']:
            drawn = state_updated['deck'].pop(0)
            state_updated[player_id].append(drawn)
       

    else:
        hand = state_updated[player_id]
        for i, c in enumerate(hand):
            if c == card:
                hand.pop(i)
                break

        if 'discards' not in state_updated:#purana top ko discard mai 
            state_updated['discards'] = []
            
        state_updated['discards'].append(state_updated['top_card'])

        state_updated['top_card'] = card


        
        if card.is_skip():
            index = PLAYERS.index(player_id)
            skip_player = PLAYERS[(index + 1) % len(PLAYERS)]


    
    return state_updated, skip_player

In [173]:
def evaluate(state, player_id, strategy= 'defensive'):
    opp_keys = [k for k in PLAYERS if k != player_id]
    
    cai = len(state[player_id])
    copp = sum(len(state[k]) for k in opp_keys) / 2.0
    s = sum(1 for c in state[player_id] if c.is_skip())

    if strategy == 'defensive':#penalise own hand harder and reward skips highly
        return 50.0 - 6.0 * cai + 2.0 * copp + 4.0 * s
    else:#moderate self-penalty adn strongly reward opponent burden
        return 50.0 - 5.0 * cai + 3.0 * copp + 2.0 * s

In [174]:
class TreeNode:
    
    def __init__(self, label, node_type, score = None):
        self.label = label
        self.node_type = node_type   # max , min ,chance,opp,leaf, terminal pruned
        self.children = []
        self.score = score
        


    
    def add_child(self, child: 'TreeNode'):
        self.children.append(child)




def print_tree(node, prefix = "",is_root= True, is_last= True, max_depth = 3,cur_depth= 0):#recursive
   
    if node.score is not None:
        scorestr = f" [{node.score:+.1f}]"
    else:
        scorestr = " "
    if is_root:
        print(f"[{node.node_type:8s}] {node.label}{scorestr}")
        childpre = ""
        
    else:
        if is_last:
            conn = "|__ "
        else:
            conn = "|-- "
        print(f"{prefix}{conn}[{node.node_type:8s}] {node.label}{scorestr}")
        
        if is_last:
            childpre = prefix + "    "
        else:
            childpre = prefix + "|   "


        
    if cur_depth >= max_depth:
        if node.children:
            print(f"{childpre}... ({len(node.children)} children not shown)")
        return



    
    for i, child in enumerate(node.children):
        print_tree(child, childpre, is_root=False,is_last=(i == len(node.children) - 1), max_depth=max_depth, cur_depth=cur_depth + 1)
print("complete process of ai decisions above")



complete process of ai decisions above


In [175]:
def minimax(state,depth, player_id, current_player,skipped, alpha = float('-inf'), beta = float('inf'), tree_build = True):
 
    for p in PLAYERS:
        if len(state[p]) == 0:
            s = (1000.0 + depth) if p == player_id else (-1000.0 - depth)  
            node = TreeNode(f"WIN({p.upper()})", "TERMINAL", s) if tree_build else None
            
            return s, None, node

    
    if depth == 0:
        s = evaluate(state, player_id, 'defensive')
        node = TreeNode("Leaf", "LEAF", s) if tree_build else None

        return s, None, node

    
    if current_player in skipped:#skip propagation
        new_skipped = skipped - {current_player}
        index = PLAYERS.index(current_player)
        next_index = PLAYERS[(index + 1) % len(PLAYERS)]
        
        return minimax(state, depth, player_id, next_index, new_skipped,alpha, beta, tree_build)

    
    valid   = get_valid_moves(state[current_player], state['top_card'])
    if valid:
        actions = valid
    else:
        actions = [None] 

    is_max    = (current_player == player_id)
    if is_max:
        node_type = "MAX"
    else:
        node_type = "MIN"
    
    index = PLAYERS.index(current_player)
    next_index = PLAYERS[(index + 1) % len(PLAYERS)]

    root_node = (TreeNode(f"{current_player.upper()} | top={state['top_card']}",  node_type) if tree_build else None)

    if is_max:
        bestscore = float('-inf')
    else:
        bestscore = float('inf')
        
    if actions:
        bestmove = actions[0]
    else:
        bestmove = None

    for i in actions:
        state_updated, skip_p = apply_move(state, current_player, i)
        if skip_p:
            new_skipped = {skip_p}
        else:
            new_skipped = set()

        child_score, _, child_node = minimax(state_updated, depth - 1, player_id, next_index, new_skipped,alpha, beta, tree_build)

        if tree_build:
            if i is not None:
                action_str = str(i)
            else:
                action_str = "Draw forcefully"
            anode = TreeNode(action_str, "ACTION", child_score)
            if child_node:
                anode.add_child(child_node) 
            root_node.add_child(anode)

        if is_max:
            if child_score > bestscore:
                bestscore = child_score
                bestmove  = i
            alpha = max(alpha, bestscore)
            
        else:
            if child_score < bestscore:
                bestscore = child_score
                bestmove  = i
                
            beta = min(beta, bestscore)

        
        # alphabeta pruning
        if beta <= alpha:
            if tree_build:
                root_node.add_child(TreeNode("(pruned)", "PRUNED"))
            break



    if tree_build:
        root_node.score = bestscore
    return bestscore, bestmove, root_node

In [176]:
def expectimax(state,depth,player_id,current_player,skipped, tree_build= True):

    for p in PLAYERS:
        if len(state[p]) == 0:
            if p == player_id:
                s = 1000.0 + depth
            else:
                s = -1000.0 - depth
            node = TreeNode(f"WIN({p.upper()})", "TERMINAL", s) if tree_build else None
            return s, None, node

    if depth == 0:
        s = evaluate(state, player_id, 'offensive')
        node = TreeNode("Leaf", "LEAF", s) if tree_build else None
        return s, None, node

    
    if current_player in skipped:
        new_skipped = skipped - {current_player}
        index = PLAYERS.index(current_player)
        nextindex = PLAYERS[(index + 1) % len(PLAYERS)]
        return expectimax(state, depth, player_id, nextindex, new_skipped, tree_build)

    valid = get_valid_moves(state[current_player], state['top_card'])
    index   = PLAYERS.index(current_player)
    nextindex   = PLAYERS[(index + 1) % len(PLAYERS)]

    
    if current_player == player_id:
        root_node = (TreeNode(f"{current_player.upper()} | top={state['top_card']}", "MAX")if tree_build else None)
        bestscore = float('-inf')
        bestmove  = None

        for card in valid:
            state_updated,skip_p = apply_move(state, current_player, card)
            if skip_p:
                new_skipped = {skip_p}
            else:
                new_skipped = set()
                        
            score, _, child_node = expectimax(state_updated, depth - 1, player_id, nextindex, new_skipped, tree_build)

            if tree_build:
                anode = TreeNode(f"Play {card}", "ACTION", score)
                if child_node:
                    anode.add_child(child_node)  
                root_node.add_child(anode)

            if score > bestscore:
                bestscore = score
                bestmove  = card

        if not valid:
            if state['deck']:
                chance_score, chance_node = _chance_node(
                    state, current_player, player_id,
                    nextindex, set(), depth, tree_build)
                if tree_build:
                    root_node.add_child(chance_node)
                if chance_score > bestscore:
                    bestscore = chance_score
                    bestmove  = None
                    
            else:
                bestscore = evaluate(state, player_id, 'offensive')

        if tree_build:
            root_node.score = bestscore

            
        return bestscore, bestmove, root_node


    
    else:
        root_node = (TreeNode( f"{current_player.upper()} | top={state['top_card']} ","OPP") if tree_build else None)

        if valid:
            actions = valid
        else:
            if state['deck']:
                actions = [None]   
            else:
                actions = []       

        if not actions:
            s = evaluate(state, player_id, 'offensive')
            if tree_build:
                root_node.score = s
                
            return s, None, root_node

        total_score = 0.0
        
        for move in actions:
            state_updated,skip_p = apply_move(state, current_player, move)
            if skip_p:
                new_skipped = {skip_p}
            else:
                new_skipped = set()
            
            score, _, child_node = expectimax(state_updated, depth - 1, player_id, nextindex, new_skipped, tree_build)
            
            total_score += score

            if tree_build:
                if move is not None:
                    action_str = str(move)
                else:
                    action_str = "Draw (forced)"
                anode = TreeNode(action_str, "OPP", score)
                if child_node:
                    anode.add_child(child_node)   
                root_node.add_child(anode)

        expected_score = total_score / len(actions)

        
        if tree_build:
            root_node.score = expected_score

            
        return expected_score, None, root_node



In [177]:

def _chance_node(state, current_player,  player_id,next_player, skipped, depth, tree_build):
    
    deck = state['deck']
    
    if tree_build:
        chance_node = TreeNode("Draw Chance", "CHANCE")
    else:
        chance_node = None

    if not deck:
        s = evaluate(state, player_id, 'offensive')
        
        if tree_build:
            chance_node.score = s
        return s, chance_node

    card_counts= defaultdict(int)
    for c in deck:
        card_counts[(c.colour, c.value)] += 1

    deck_size = len(deck)
    expected_val = 0.0

    for (colour, value), count in card_counts.items():
        prob  = count / deck_size
        drawn = Card(colour, value)

        state_updated = copy.deepcopy(state)
        state_updated[current_player].append(drawn)
        
        for i, c in enumerate(state_updated['deck']):
            if c == drawn:
                state_updated['deck'].pop(i)
                break

        score, _, child_node = expectimax(state_updated, depth - 1, player_id, next_player, skipped, tree_build)
        expected_val += prob * score

        if tree_build:
            cnode = TreeNode(f"Draw {drawn} (p={prob:.2f})", "  CHANCE", score)
            if child_node:
                cnode.add_child(child_node)   
            chance_node.add_child(cnode)

    if tree_build:
        chance_node.score = expected_val

        
    return expected_val, chance_node

In [178]:
class GAME_UNO:
    
    def __init__(self, mode = 'simulation', seed = 42):
        random.seed(seed)
        self.mode = mode
        self.seed = seed         

        deck = generate_deck()
        self.state = {'p1':       [deck.pop() for _ in range(5)],
            'p2':       [deck.pop() for _ in range(5)],
            'p3':       [deck.pop() for _ in range(5)],
            'top_card': deck.pop(),
            'deck':     deck,
            'discards': [],
            
        }

        
        attempts = 0
        while (self.state['top_card'].is_skip() and self.state['deck'] and attempts < 10):
            self.state['deck'].insert(0, self.state['top_card'])
            self.state['top_card'] = self.state['deck'].pop()
            attempts += 1

        self.current_player = 'p1'
        self.skipped = set()
        self.winner = None
        self.move_history = []
        self.turn = 0
        self.game_over = False
        

        self.display()


    def display(self):
        print("\n" + "-" * 65)
        print("                       UNO                    ")
        print("-" * 65)
        print(f"  Mode : {'Simulation (AI all players)' if self.mode == 'simulation' else 'Manual (You are P3)'}")
        
        print(f"  Seed : {self.seed}")   
        print(f"\n  Top Card  : {self.state['top_card']}")
        
        for p in PLAYERS:
            print(f"  {p.upper()} hand: {self.state[p]}")
        print(f"  Deck size : {len(self.state['deck'])} cards")
        
        print("_" * 65)

    def show_state(self):
        disc = len(self.state.get('discards', []))
        print(f"\n{'-'*65}")
        print(f"  Turn {self.turn:3d} | {self.current_player.upper()}'s move" f"  Top: {self.state['top_card']}")
        print(f"{'-'*65}")
        for p in PLAYERS:
            if p == self.current_player:
                marker = " <"
            else:
                marker = "  "
            
            print(f"  {PLAYER_LABELS[p]:<32} {len(self.state[p]):2d} cards  {self.state[p]}{marker}")

            
        print(f"  Deck: {len(self.state['deck'])} | Discards: {disc}", end="")
              
        if self.skipped:
            print(f"  | Skipped: {self.skipped}", end="")

            
        print()

   
    def show_depth1_scores(self, player: str, strategy: str):
       
        valid   = get_valid_moves(self.state[player], self.state['top_card'])
        if valid:
            actions = valid
        else:
            actions = [None]

        if strategy == 'defensive':
            label = "Heuristic score"
        else:
            label = "Expected score"

        print(f"\n  Top card: {self.state['top_card']}")
        print(f"  {player.upper()} hand:")
        
        for c in self.state[player]:
            print(f"  {c}")
            
        print(f"\n  All actions considered at depth 1:")
        
        for action in actions:
            tmp, _ = apply_move(self.state, player, action)
            score  = evaluate(tmp, player, strategy)
            if action is not None:
                label1 = str(action)
            else:
                label1 = "Draw (forcefully )"
                
            print(f"    {label1:<26}  {label}: {score:.1f}")

    
    def _p1_move(self):
        print("\n P1 MINIMAX DEFENSIVE")
        self.show_depth1_scores('p1', 'defensive')

        score, bestmove, tree = minimax(self.state, depth=3,player_id='p1', current_player='p1',skipped=self.skipped, alpha=float('-inf'), beta=float('inf'),tree_build=True)

        if bestmove is not None:
            label = str(bestmove)
        else:
            label = "Draw (forcefullyu)"
        
        print(f"\n  -- Minimax decision   depth 3: {label}  / score = {score:.1f}")
        print("\n  Minimax search tree (depth 2):   ")
        
        if tree:
            print_tree(tree, max_depth=2)
        print()

        
        return bestmove

    
    def _p2_move(self) :
        print("\n  P2 EXPECTIMAX OFFENSIVE")
        valid = get_valid_moves(self.state['p2'], self.state['top_card'])

        print(f"\n  Top card: {self.state['top_card']}")
        print("  P2 hand:")
        
        for c in self.state['p2']:
            print(f"  {c}")
        print("\n  All actions considered at depth 1:")

        if valid:
            for card in valid:
                tmp, _ = apply_move(self.state, 'p2', card)
                s = evaluate(tmp, 'p2', 'offensive')
                
                print(f" Play {str(card):<22}  Expected score: {s:.1f}")
        else:
            if self.state['deck']:
                cs, _ = _chance_node(self.state, 'p2', 'p2', 'p3', set(), 1, False)
                print(f" Draw (forced):   Expected score: {cs:.2f}")

        score, bestmove, tree = expectimax(self.state, depth=3,player_id='p2', current_player='p2', skipped=self.skipped,tree_build=True)

        if bestmove is not None:
            label = str(bestmove)
        else:
            label = "Draw (forcefully)"
            
        print(f"\n -- Expectimax decision (depth 3): {label}  /  score = {score:.1f}")
        print("\n Expectimax search tree (depth-2 ):")
        if tree:
            print_tree(tree, max_depth=2)
            
        print()
        
        return bestmove

        

   
    def _p3_manual(self):
        print("\n  P3 HUMAN or YOU")
        print(f"Your hand : {self.state['p3']}")
        print(f"Top card  : {self.state['top_card']}")
        valid = get_valid_moves(self.state['p3'], self.state['top_card'])

        if not valid:
            print(" NO valid moves u must draw a card")
            return None

        print(" Valid moves:")
        for i, card in enumerate(valid):
            print(f"    {i}: {card}")
        

        while True:
            try:
                choice = int(input("  Enter index: "))
                if 0 <= choice < len(valid):
                    return valid[choice]
                print("Invalid Try again")
                
            except ValueError:
                print("Enter a number")

    
    def _p3_sim(self):
        print("\n P3 MINIMAX SIMULATION")
        self.show_depth1_scores('p3', 'defensive')

        score, bestmove, _ = minimax(self.state, depth=3, player_id='p3', current_player='p3', skipped=self.skipped, alpha=float('-inf'), beta=float('inf'), tree_build=False)

        if bestmove is not None:
            label = str(bestmove)
        else:
            label = "Draw (forcefully)"
        print(f"\n  --- Minimax decision (depth 3): {label}  |  score = {score:.1f}")

        
        return bestmove

    
    def _run(self, player, card):
        old_len = len(self.state[player])
        self.state, skip_p = apply_move(self.state, player, card)

        if card is None:
            if len(self.state[player]) > old_len:
                drawn = self.state[player][-1]
                print(f"\n  >> {player.upper()} DRAWS -> {drawn}")
            else:
                print(f"\n  >> {player.upper()} tried to draw but deck is empty!")
                
        else:
            print(f"\n  >> {player.upper()} PLAYS -> {card}")

        if skip_p:
            self.skipped.add(skip_p)
            print(f"  [SKIP] {skip_p.upper()} will be SKIPPED next turn!!! womp")

        self.move_history.append({'turn':     self.turn,
            'player':   player,
            'action':   str(card) if card is not None else 'Draw',
            'top_card': str(self.state['top_card']),})

    
    def play_turn(self):
        self.turn += 1
        player = self.current_player
        self.show_state()

        
        if player in self.skipped:#skip
            print(f"\n  [SKIP] {player.upper()} is SKIPPED this turn!!! womp")
            self.skipped.discard(player)
            index = PLAYERS.index(player)
            
            self.current_player = PLAYERS[(index + 1) % len(PLAYERS)]
            return

    
        if player == 'p1':
            card = self._p1_move()
        elif player == 'p2':
            card = self._p2_move()
        else:
            if self.mode == 'manual':
                card = self._p3_manual()
            else:
                card = self._p3_sim()

        self._run(player, card)

        
        for p in PLAYERS:
            if len(self.state[p]) == 0:
                self.game_over = True
                self.winner    = p
                print(f"\n{'_'*65}")
                print(f"  GAME OVER {PLAYER_LABELS[p]} WINSSS!")
                print("-" * 65)
                return

        index = PLAYERS.index(player)
        self.current_player = PLAYERS[(index + 1) % len(PLAYERS)]

   
    def play(self, max_turns = 200):#complete game
        while not self.game_over and self.turn < max_turns:
            self.play_turn()
            
            if self.mode == 'manual' and not self.game_over:
                input("\nPRESS ENTER TO NEXT TURN ")

        if not self.game_over:
            print(f"\n  Game ended after {max_turns} turns (no winner) WOMPP")
            
            for p in PLAYERS:
                print(f" {p.upper()}: {len(self.state[p])} cards")


        
        return self.winner

In [179]:
import io, contextlib

In [180]:
def compare_algorithms(n_games = 5) :
    
    print("\n" + "_" * 65)
    print(f"  ALGORITHM COMPARISON  ({n_games} games)")
    print("_" * 65)

    wins = {'p1': 0, 'p2': 0, 'p3': 0, 'timeout': 0}

    for i in range(n_games):
        seed = i * 137 + 42
        print(f"\n  Game {i+1}/{n_games}  (seed={seed})", end="  ")
        
        buf = io.StringIO()
        
        with contextlib.redirect_stdout(buf):#run game in memory not terminal bohat zada but agar dekhna  comment 
            game   = UNOGame(mode='simulation', seed=seed)
            winner = game.play(max_turns=150)
            
        if winner:
            wins[winner] += 1
            
            print(f"Winner: {PLAYER_LABELS[winner]}")
            
        else:
            wins['timeout'] += 1
            print("No winner (timeout)")

    
    print("\n" + "-" * 65)
    print("  RESULTS   ")
    print("-" * 65)
    print(f"P1 Minimax(Defensive) :  {wins['p1']:2d} / {n_games}")
    print(f"P2 Expectimax (Offensive) :  {wins['p2']:2d} / {n_games}")
    print(f"P3 Minimax (Simulation) :  {wins['p3']:2d} / {n_games}")
    print(f"Timeout: {wins['timeout']:2d} / {n_games}")


    best = max(['p1', 'p2', 'p3'], key=lambda p: wins[p])
    
    if wins[best] == 0:
        verdict = "No clear winner all games timed out"
    elif best == 'p2':
        verdict = "Expectimax (Offensive) performed best "
    elif best == 'p1':
        verdict = "Minimax Defensive (P1) performed best "
    else:
        verdict = "Minimax Simulation (P3) performed best "

        
    print(f"\n  Conclusion: {verdict}")


    return wins

In [182]:
print("\n" + "_" * 65)

print("\n                 UNO — MENU            ")
print("_" * 65)
print("  1. Simulation Mode- All 3 players are AI")
print("  2. Manual Mode  - You play as Player 3")
print("  3. Tree Demo - Search tree on fixed example")
print("  4. Compare  - 5 games + algorithm analysis")

print("_" * 65)

try:
    choice = input("\n  Enter choice [1-5] (default=1): ").strip() or '1'
except (EOFError, KeyboardInterrupt):
    choice = '1'

if choice == '1':
    game = GAME_UNO(mode='simulation', seed=random.randint(1, 1000000))
    game.play(max_turns=150)

elif choice == '2':
    game = UNOGame(mode='manual', seed=random.randint(1, 1000000))
    game.play(max_turns=150)
    
elif choice == '3':
    run_tree_demo()
    
elif choice == '4':
    compare_algorithms(n_games=5)


else:
    print("Invalid choice Running simulation as defualt")
    game = UNOGame(mode='simulation', seed=random.randint(1, 1000000))
    game.play(max_turns=150)





_________________________________________________________________

                 UNO — MENU            
_________________________________________________________________
  1. Simulation Mode- All 3 players are AI
  2. Manual Mode  - You play as Player 3
  3. Tree Demo - Search tree on fixed example
  4. Compare  - 5 games + algorithm analysis
_________________________________________________________________



  Enter choice [1-5] (default=1):  4



_________________________________________________________________
  ALGORITHM COMPARISON  (5 games)
_________________________________________________________________

  Game 1/5  (seed=42)  

TypeError: apply_move() takes 2 positional arguments but 3 were given

<h1>EVALUATION FUNCTION EXPLANATION</h1>

Base formula (from assignment) Score = 50 − 5x(CAI) + 2x(Copp) + 3x(S)
    CAI  = number of cards in the AI player's hand  (lower -> better)
    Copp = average cards held by the two opponents   (higher -> better)
    S    = number of Skip cards in the AI's hand     (higher -> better)

The formula tuned differently per strategy
DEFENSIVE (Player 1 Minimax)
  - owncard penalty raised  (−6 instead of −5) defensive play is obsessed with keeping the hand small
  - skip reward raised (+4 instead of +3) skip cards are gold they freeze opponents at critical moments
  - opponent burden kept moderate (+2) we care less about loading opponents and more about protecting ourself
  Formula -> 50 − 6xCAI + 2xCopp + 4xS



OFFENSIVE (P2 Expectimax)
  - owncard penalty standard (−5)we still want to shed cards but not at the expense of loading opponents
  - Opponent burden raised (+3 instead of +2)offensive play tries to force opponents to have MORE cards (draw situations)
  -Skip reward lowered (+2)skips are useful but we prefer getting rid of cards over delaying opps by skipping them
  Formula -> 50 − 5xCAI + 3xCopp + 2xS

#### <h1>ALGORITHM ANALYSIS</h1>


  **Minimax Defensive  (P1 & P3)**
  Strategy  : Prevent opponents from winning save Skips for crises
  Assumption: All opponents play OPTIMALLY against us (worst case)
  Search    : Alpha-beta pruning prunes branches when beta <= alpha
  Weights   : -6*own  +2*opp  +4*skip
  Strength  : Robust when opponents are smart
  Weakness  : Overly pessimistic when opponents play randomly
              may stall because it holds Skips too long

  **Expectimax Offensive  (P2)**
  Strategy  : Aggressive card-shedding; exploit the draw deck
  Assumption: Draw is a CHANCE NODE with exact deck probabilities
              Opponents modelled as average over ALL legal moves
              (deterministic expected value -- not random sampling)
  Weights   : -5*own  +3*opp  +2*skip
  Strength  : Explicitly models randomness -- best fit for UNO
  Weakness  : Opponent "average" assumption may be optimistic



**EXPECTIMAX PERFORMED BEST**

  **Why Expectimax tends to win in UNO**
  1. UNO's draw mechanic is INHERENTLY RANDOM expectimax models this correctly via chance nodes Minimax ignores it
  2. offensive strategy sheds cards faster reaching 0 sooner
  3. minimax's pessimistic min assumption punishes itself when real opponents are not adversarial
  4. after fixing draw only whe forced  expectimax's chancenode probability calculations are fully accurate